# 05 — Final evaluation: random test, leave-one-donor-out, leave-one-site-out

The design, metrics and expectations were fixed in the README **before** any result: the pre-registration
addendum, commit `fa39914`. This notebook runs that design once.

* **R**: random split, test part, evaluated **once**. Ridge 3a is refit on the training part; the MLP
  uses the three Phase 4 checkpoints (seeds 0-2).
* **D**: leave-one-donor-out, 9 folds. **C**: leave-one-site-out, 4 folds. Frozen hyperparameters; every
  learned step (HVG, scaling, PCA, ridge, target scaling) is refit on the fold's training cells only.
  MLP: 1 seed, early stopping on an inner 15% of the fold's training cells.
* Metric 1: within cell type x sample Pearson. Metric 2: standardized bias per (cell type x test sample),
  cell-weighted mean |b| and shift captured = 1 - Σ|b|(model) / Σ|b|(cell-type mean); cell types missing
  from training are excluded and counted. Metric 3: pooled within cell type Pearson, only for test sets
  with >= 2 samples.
* After this run **no setting and no mapping is changed**.

Smoke-test mode (`CITESEQ_SUBSET`) uses only random-split train/val cells, with **shuffled protein
targets**. It exercises the code without producing any real result.

In [ ]:
import os, sys, time, json, hashlib
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from citeseq.paths import RESULTS_DIR, FIGURES_DIR, PROCESSED_DIR
from citeseq.data_prep import load_all
from citeseq.splits import load_split, make_random_split, group_folds
from citeseq.phase5 import fit_predict_fold, group_bias, weighted_abs_bias, shift_captured
from citeseq.correlation import within_group_pearson, summarize_within
from citeseq.metrics import evaluate
from citeseq import plotting as P

torch.set_num_threads(8)
SUBSET = int(os.environ.get("CITESEQ_SUBSET", 0)) or None
MAX_EPOCHS = int(os.environ.get("CITESEQ_MAX_EPOCHS", 200))
MIN_CELLS = 50
cfg3 = json.loads((RESULTS_DIR / "phase3_frozen_configs.json").read_text())
cfg4 = json.loads((RESULTS_DIR / "phase4_frozen_config.json").read_text())
OUT = PROCESSED_DIR / ("phase5" if SUBSET is None else "test_phase5")
OUT.mkdir(parents=True, exist_ok=True)
OUT_FIG = FIGURES_DIR if SUBSET is None else OUT / "figures"
OUT_FIG.mkdir(parents=True, exist_ok=True)
P.set_style()
pd.set_option("display.float_format", "{:.3f}".format)
print("subset:", SUBSET, "| max epochs:", MAX_EPOCHS)

## 1. Data and splits

In [ ]:
split_full = None
mask = None
if SUBSET:
    # smoke test: only random-split train/val cells (never the test part), targets shuffled
    from citeseq.paths import CITE_H5AD
    from citeseq.io import _read_dataframe
    import h5py
    with h5py.File(CITE_H5AD) as f:
        names = _read_dataframe(f["obs"]).index
    split_full = load_split(RESULTS_DIR / "splits/random_split.csv.gz", names)
    rng = np.random.default_rng(0)
    pool = np.flatnonzero((split_full != "test").to_numpy())
    mask = np.zeros(len(names), bool); mask[rng.choice(pool, SUBSET, replace=False)] = True
d = load_all(cell_mask=mask)
X, Y, R, obs, proteins, pmap, genes = d["X"], d["Y"], d["R"], d["obs"], d["proteins"], d["pmap"], d["gene_names"]
if SUBSET:
    Y = Y[np.random.default_rng(1).permutation(len(Y))]          # no real signal in smoke tests
    split = make_random_split(obs, seed=0)
else:
    split = load_split(RESULTS_DIR / "splits/random_split.csv.gz", obs.index)
primary_cols = np.flatnonzero((pmap["analysis_set"] == "primary").to_numpy())
ct = obs["cell_type"].astype(str).to_numpy()
sample = obs["batch"].astype(str).to_numpy()
donor = obs["DonorNumber"].astype(str).to_numpy()
site = obs["Site"].astype(str).to_numpy()
type_sample = np.char.add(np.char.add(ct.astype(str), " | "), sample.astype(str))

folds = {("R", "random"): ((split == "train").to_numpy(), (split == "test").to_numpy())}
for g, te in group_folds(obs, "DonorNumber").items():
    folds[("D", g)] = (~te, te)
for g, te in group_folds(obs, "Site").items():
    folds[("C", g)] = (~te, te)
print(X.shape, "| folds:", [f"{k[0]}:{k[1]}" for k in folds])

## 2. Fit and predict per fold (saved to disk; a finished fold is re-used)

In [ ]:
def p4_ckpt(seed):
    b = cfg4["best"]
    key = json.dumps([b["hidden"], b["dropout"], b["weight_decay"], seed, 200, 15])
    return PROCESSED_DIR / "phase4_checkpoints" / f"mlp_{hashlib.md5(key.encode()).hexdigest()[:10]}.pt"

results = {}
t_all = time.time()
for (sp_name, fold), (train, test) in folds.items():
    f = OUT / f"{sp_name}_{fold}.npz"
    if f.exists():
        z = np.load(f, allow_pickle=True)
        results[(sp_name, fold)] = {"preds": z["preds"].item(), "state": z["state"].item()}
        print(f"{sp_name}:{fold:<8} loaded")
        continue
    t = time.time()
    if sp_name == "R":
        kw = dict(mlp_seeds=(0, 1, 2), mlp_val=(split == "val").to_numpy(),
                  checkpoint_fn=None if SUBSET else p4_ckpt)
    else:
        kw = dict(mlp_seeds=(0,), checkpoint_fn=lambda s, sp_name=sp_name, fold=fold: OUT / f"mlp_{sp_name}_{fold}_s{s}.pt")
    preds, state = fit_predict_fold(X, Y, R, obs, genes, train, test, cfg3, cfg4, max_epochs=MAX_EPOCHS, **kw)
    if sp_name == "R" and not SUBSET:
        assert all(v["loaded"] for v in state["mlp"].values()), "Phase 4 checkpoints were not re-used"
    np.savez_compressed(f, preds=np.array(preds, dtype=object), state=np.array(state, dtype=object))
    results[(sp_name, fold)] = {"preds": preds, "state": state}
    print(f"{sp_name}:{fold:<8} train {train.sum():>6,} test {test.sum():>6,} | MLP fit/stop {state['n_mlp_fit']:,}/{state['n_mlp_stop']:,} "
          f"| MLP {state['mlp']} | {(time.time() - t) / 60:.1f} min")
print(f"all folds: {(time.time() - t_all) / 60:.1f} min")

## 3. Metrics

In [ ]:
def mlp_keys(preds):
    return sorted(k for k in preds if k.startswith("MLP"))

def models_of(preds):
    return ["cell-type mean", "own gene", "ridge 3a"] + mlp_keys(preds)

rows, bias_long, r_long, excl_rows = [], [], [], []
for (sp_name, fold), res in results.items():
    train, test = folds[(sp_name, fold)]
    preds, state = res["preds"], res["state"]
    te = np.flatnonzero(test)
    yt = Y[te]
    seen = np.isin(ct[te], state["train_types"])
    excl_types = sorted(set(ct[te][~seen]))
    excl_rows.append({"split": sp_name, "fold": fold, "excluded_types": len(excl_types),
                      "excluded_cells": int((~seen).sum()), "types": ", ".join(excl_types)})
    # report groups: C folds are split into donor1 vs other donors
    subsets = {"all": np.ones(len(te), bool)}
    if sp_name == "C":
        subsets = {"donor1": donor[te] == "donor1", "other donors": donor[te] != "donor1"}
    for sub_name, sub in subsets.items():
        ref = group_bias(yt, preds["cell-type mean"], type_sample[te], state["sd_train"], MIN_CELLS, keep=sub & seen)
        n_samples = len(set(sample[te][sub]))
        for m in models_of(preds):
            p = preds[m]
            lb = group_bias(yt, p, type_sample[te], state["sd_train"], MIN_CELLS, keep=sub & seen)
            bias_long.append(lb.assign(split=sp_name, fold=fold, subset=sub_name, model=m))
            if sub.sum() and m != "cell-type mean":
                rl = within_group_pearson(p[sub], yt[sub], type_sample[te][sub], MIN_CELLS)
                r_long.append(rl.assign(split=sp_name, fold=fold, subset=sub_name, model=m))
            e_ts = evaluate(yt[sub], p[sub], type_sample[te][sub], proteins, MIN_CELLS)
            e_t = evaluate(yt[sub], p[sub], ct[te][sub], proteins, MIN_CELLS)
            rows.append({"split": sp_name, "fold": fold, "subset": sub_name, "model": m, "cells": int(sub.sum()),
                         "samples": n_samples,
                         "M1 within type x sample": e_ts["within_w"].iloc[primary_cols].median(),
                         "n type x sample groups": int(e_ts["n_types"].median()),
                         "M2 mean |b|": weighted_abs_bias(lb, primary_cols),
                         "M2 shift captured": shift_captured(lb, ref, primary_cols) if m != "cell-type mean" else np.nan,
                         "M3 pooled within type": e_t["within_w"].iloc[primary_cols].median() if n_samples >= 2 else np.nan,
                         "pearson all cells": e_t["pearson_all"].iloc[primary_cols].median()})
per_fold = pd.DataFrame(rows)
excluded = pd.DataFrame(excl_rows)
bias_long = pd.concat(bias_long, ignore_index=True)
r_long = pd.concat(r_long, ignore_index=True)
print("cell types excluded from metric 2 (absent from the fold's training cells):")
excluded[excluded["excluded_types"] > 0]

### Report groups

* **R test**: random split; samples seen in training.
* **D, donors 2-9**: unseen donor; its site is seen (through donor1 and the other donor there). 8 folds
  pooled.
* **D, donor1**: unseen donor (4 samples), all sites seen through other donors.
* **C, donor1 cells**: donor seen, site unseen. **C, other donors**: donor and site unseen. 4 folds pooled.

Pooled numbers combine the per-fold (group, protein) pairs. Metric 1 is summarized over the pooled groups,
metric 2 over the pooled sums, and metric 3 is reported per fold.

In [ ]:
def report_group(sp_name, folds_sel, subset):
    sel_b = (bias_long["split"] == sp_name) & bias_long["fold"].isin(folds_sel) & (bias_long["subset"] == subset)
    sel_r = (r_long["split"] == sp_name) & r_long["fold"].isin(folds_sel) & (r_long["subset"] == subset)
    out = {}
    lb_ref = bias_long[sel_b & (bias_long["model"] == "cell-type mean")]
    lb_ref_u = lb_ref.assign(group=lb_ref["fold"] + "/" + lb_ref["group"])
    for m in sorted(bias_long.loc[sel_b, "model"].unique()):
        lb = bias_long[sel_b & (bias_long["model"] == m)]
        lb_u = lb.assign(group=lb["fold"] + "/" + lb["group"])
        rl = r_long[sel_r & (r_long["model"] == m)]
        m1 = np.nan
        if len(rl):
            rl_u = rl.assign(group=rl["fold"] + "/" + rl["group"])
            s = summarize_within(rl_u, len(proteins))
            m1 = s["weighted_mean"].iloc[primary_cols].median()
        out[m] = {"M1 within type x sample": m1,
                  "M2 mean |b|": weighted_abs_bias(lb_u, primary_cols),
                  "M2 shift captured": shift_captured(lb_u[["group", "j", "n", "b"]], lb_ref_u[["group", "j", "n", "b"]], primary_cols)
                                       if m != "cell-type mean" else np.nan,
                  "type x sample groups": lb_u["group"].nunique()}
    return pd.DataFrame(out).T

donors29 = [f"donor{i}" for i in range(2, 10)]
sites = sorted(set(site))
groups_def = {
    "R test": ("R", ["random"], "all"),
    "D donors 2-9": ("D", donors29, "all"),
    "D donor1": ("D", ["donor1"], "all"),
    "C donor1 cells": ("C", sites, "donor1"),
    "C other donors": ("C", sites, "other donors"),
}
report = pd.concat({k: report_group(*v) for k, v in groups_def.items()}, names=["report group", "model"])
report

In [ ]:
# MLP on R: mean ± SD over the 3 seeds (D and C use seed 0 only)
r_mlp = report.loc["R test"].loc[[k for k in report.loc["R test"].index if k.startswith("MLP")]]
print("R test, MLP over seeds 0-2: mean"); print(r_mlp.mean(numeric_only=True).round(4).to_string())
print("SD"); print(r_mlp.std(numeric_only=True, ddof=1).round(4).to_string())

In [ ]:
print("Metric 3, pooled within cell type (test sets with >= 2 samples), and Pearson over all cells:")
m3 = per_fold[per_fold["samples"] >= 2][["split", "fold", "subset", "model", "samples", "cells",
                                          "M3 pooled within type", "pearson all cells"]]
m3

In [ ]:
print("Per fold, ridge 3a and MLP seed 0:")
per_fold[per_fold["model"].isin(["ridge 3a", "MLP s0", "cell-type mean"])][
    ["split", "fold", "subset", "model", "cells", "samples", "M1 within type x sample", "M2 mean |b|", "M2 shift captured"]]

## 4. Figures

In [ ]:
order = list(groups_def)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
series = [("cell-type mean", P.CONTEXT, "o"), ("ridge 3a", P.BLUE, "s"), ("MLP s0", P.ORANGE, "D")]
x = np.arange(len(order))
for ax, metric, title in zip(axes, ["M2 mean |b|", "M2 shift captured", "M1 within type x sample"],
                             ["Metric 2: mean |standardized bias|", "Metric 2: shift captured", "Metric 1: within type x sample r"]):
    for (m, colr, mk), off in zip(series, (-0.18, 0, 0.18)):
        vals = [report.loc[(g, m), metric] if (g, m) in report.index else np.nan for g in order]
        if np.all(np.isnan(np.asarray(vals, dtype=float))):
            continue
        ax.scatter(x + off, vals, color=colr, marker=mk, s=40, label=m, edgecolors=P.SURFACE, linewidths=0.8, zorder=3)
    ax.set_xticks(x, order, rotation=25, ha="right"); ax.set_title(title)
    ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False, fontsize=8)
axes[1].axhline(0, color=P.MUTED, lw=1)
fig.tight_layout()
fig.savefig(OUT_FIG / "phase5_bias_shift.png")

In [ ]:
if SUBSET is None:
    per_fold.to_csv(RESULTS_DIR / "phase5_per_fold.csv", index=False)
    report.reset_index().to_csv(RESULTS_DIR / "phase5_report_groups.csv", index=False)
    excluded.to_csv(RESULTS_DIR / "phase5_excluded_cell_types.csv", index=False)
import resource, platform
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
print(f"peak memory: {(peak / 1e9 if platform.system() == 'Darwin' else peak * 1024 / 1e9):.2f} GB")